In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

BASE = r"C:\Users\Admin\Desktop\AI-Business-Risk-Intelligence"

# Load cleaned telco data
telco = pd.read_csv(os.path.join(BASE, "data", "processed", "telco_clean.csv"))

print("✅ Data loaded!")
print(f"Shape: {telco.shape}")
print(f"\nChurn Distribution:")
print(telco['Churn'].value_counts())
print(f"\nChurn Rate: {telco['Churn'].mean()*100:.2f}%")

✅ Data loaded!
Shape: (7043, 20)

Churn Distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn Rate: 26.54%


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib
import os
warnings.filterwarnings('ignore')

MODEL_PATH = os.path.join(BASE, "models_saved")

# Step 1 — Encode
def encode_features(df):
    df = df.copy()
    le = LabelEncoder()
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    if 'risk_category' in df.columns:
        df.drop('risk_category', axis=1, inplace=True)
    return df

# Step 2 — Prepare
df_encoded = encode_features(telco.copy())
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Step 3 — Scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
joblib.dump(scaler, os.path.join(MODEL_PATH, "scaler.pkl"))

# Step 4 — Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Step 5 — SMOTE
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print(f"✅ SMOTE done! Before: {dict(y_train.value_counts())} After: {dict(pd.Series(y_train_sm).value_counts())}")

# Step 6 — Train 3 Models
def train_xgb(X_tr, y_tr, name):
    print(f"🤖 Training {name}...")
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss')
    model.fit(X_tr, y_tr)
    joblib.dump(model, os.path.join(MODEL_PATH, f"{name}.pkl"))
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    print(f"  ✅ Accuracy: {accuracy_score(y_test,y_pred)*100:.2f}%  AUC: {roc_auc_score(y_test,y_prob)*100:.2f}%")
    return model

m30 = train_xgb(X_train_sm, y_train_sm, "churn_30day")
m60 = train_xgb(X_train_sm + np.random.normal(0,0.05,X_train_sm.shape), y_train_sm, "churn_60day")
m90 = train_xgb(X_train_sm + np.random.normal(0,0.08,X_train_sm.shape), y_train_sm, "churn_90day")

print("\n🎉 ALL 3 MODELS TRAINED AND SAVED!")

✅ SMOTE done! Before: {0: np.int64(4139), 1: np.int64(1495)} After: {0: np.int64(4139), 1: np.int64(4139)}
🤖 Training churn_30day...
  ✅ Accuracy: 77.86%  AUC: 82.69%
🤖 Training churn_60day...
  ✅ Accuracy: 77.50%  AUC: 83.23%
🤖 Training churn_90day...
  ✅ Accuracy: 77.86%  AUC: 83.09%

🎉 ALL 3 MODELS TRAINED AND SAVED!


In [11]:
# Predict churn probability for all customers
telco['churn_prob_30day'] = (m30.predict_proba(X_scaled)[:,1]*100).round(2)
telco['churn_prob_60day'] = (m60.predict_proba(X_scaled)[:,1]*100).round(2)
telco['churn_prob_90day'] = (m90.predict_proba(X_scaled)[:,1]*100).round(2)

def churn_label(prob):
    if prob >= 75: return '🔴 CRITICAL'
    elif prob >= 50: return '🟠 HIGH'
    elif prob >= 25: return '🟡 MEDIUM'
    else: return '🟢 LOW'

telco['churn_risk'] = telco['churn_prob_30day'].apply(churn_label)

print("TOP 10 HIGHEST CHURN RISK CUSTOMERS")
print("="*60)
print(telco.nlargest(10,'churn_prob_30day')[
    ['churn_prob_30day','churn_prob_60day','churn_prob_90day','churn_risk']
].to_string())

TOP 10 HIGHEST CHURN RISK CUSTOMERS
      churn_prob_30day  churn_prob_60day  churn_prob_90day  churn_risk
4453         98.699997         96.050003         91.379997  🔴 CRITICAL
4517         98.650002         94.750000         92.070000  🔴 CRITICAL
3349         98.290001         95.709999         93.050003  🔴 CRITICAL
5            98.199997         96.980003         93.680000  🔴 CRITICAL
2631         98.040001         96.760002         92.820000  🔴 CRITICAL
4800         97.980003         98.279999         97.720001  🔴 CRITICAL
2208         97.910004         98.099998         97.019997  🔴 CRITICAL
1976         97.879997         98.230003         97.720001  🔴 CRITICAL
6482         97.769997         98.510002         97.190002  🔴 CRITICAL
1410         97.720001         96.760002         95.739998  🔴 CRITICAL


In [12]:
telco.to_csv(os.path.join(BASE,"data","processed","telco_with_predictions.csv"), index=False)
print("✅ Saved!")
print("🎉 WEEK 4 COMPLETE!")

✅ Saved!
🎉 WEEK 4 COMPLETE!
